In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
!pip install ultralytics
!pip install torchreid
!pip install motmetrics
!pip install gdown

In [3]:
!git clone https://github.com/nwojke/deep_sort.git

fatal: destination path 'deep_sort' already exists and is not an empty directory.


In [4]:
%cd deep_sort
!pip install -r requirements.txt
%cd ..

/content/deep_sort
/content


In [5]:

import os
import sys
sys.path.append("deep_sort")
print("DeepSORT path added")

DeepSORT path added


In [6]:
import cv2
import torch
import numpy as np
import time
# from ultralytics import YOLO


In [7]:
# Клонируем YOLOv5, если папки ещё нет
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git
    %cd yolov5
    !pip install -r requirements.txt --quiet
    %cd ..

# Добавляем yolov5 в sys.path, чтобы импорты работали
if 'yolov5' not in sys.path:
    sys.path.insert(0, os.path.abspath('yolov5'))

In [8]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

PyTorch: 2.11.0+cpu
CUDA available: False
Using device: cpu


In [9]:
folders = [
    "videos",
    "results",
    "overlays",
    "metrics",
    "configs"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
print("Folders created")

Folders created


#2 — Detectors

Наша цель — чтобы любой детектор возвращал одинаковый формат:

[
    [x1, y1, x2, y2, confidence],
    ...
]

Мы будем брать только класс person (class id = 0).

In [10]:
!pip install onnxruntime --quiet


In [11]:
import torch
from yolov5.models.common import DetectMultiBackend
from yolov5.utils.general import non_max_suppression
from yolov5.utils.torch_utils import select_device
import cv2
import numpy as np
import onnxruntime as ort
# from mmdet.apis import inference_detector, init_detector

In [32]:
#Базовый класс детектора
class Detector:
    def detect(self, frame):
        raise NotImplementedError

In [33]:
from ultralytics import YOLO

class YOLOv5Detector(Detector):
    def __init__(self, model_name="yolov5n.pt", conf=0.5, device=DEVICE):
        self.model = YOLO(model_name)  # ultralytics wrapper
        self.conf = conf
        self.device = device

    def detect(self, frame):
        result = self.model(frame, conf=self.conf, device=self.device, verbose=False)[0]

        detections = []
        for box in result.boxes:
            if int(box.cls[0]) != 0:  # person class
                continue

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

            detections.append([
                int(x1), int(y1), int(x2), int(y2), float(box.conf[0])
            ])

        return detections

In [34]:
import onnxruntime as ort
import numpy as np
import cv2

class NanoDetDetector(Detector):
    def __init__(self, model_path="nanodet_tiny.onnx", conf=0.5, input_size=416):
        self.session = ort.InferenceSession(model_path)
        self.conf = conf
        self.input_size = input_size

    def detect(self, frame):
        img = cv2.resize(frame, (self.input_size, self.input_size))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))[None, ...]

        outputs = self.session.run(None, {"input": img})

        # ⚠️ тут зависит от модели (оставляем как заглушку логики парсинга)
        detections = []

        # TODO: распарсить outputs в формат [x1,y1,x2,y2,score]

        return detections

In [12]:
# #Базовый класс детектора
# class Detector:
#     def detect(self, frame):
#         raise NotImplementedError

# # YOLO Detector
# class YOLOv5Detector:
#     def __init__(self, model_path, conf=0.5, device=None):
#         if device is None:
#             device = select_device('0' if torch.cuda.is_available() else 'cpu')
#         self.model = DetectMultiBackend(
#             weights=[model_path],
#             device=device,
#             dnn=False,
#             fp16=False
#         )
#         self.conf = conf
#         self.device = device
#         # В v7+ stride — это int
#         self.stride = int(self.model.stride)
#         self.names = self.model.names

#     def detect(self, frame):
#         """
#         frame: numpy array (H, W, C), BGR (OpenCV)
#         returns: list of [x1, y1, x2, y2, conf]
#         """
#         h, w = frame.shape[:2]

#         # Конвертируем BGR -> RGB и делаем тензор [1, C, H, W]
#         img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         img = img.transpose(2, 0, 1)          # HWC -> CHW
#         img = torch.from_numpy(img).float()   # uint8 -> float
#         img /= 255.0                          # нормализация 0–1
#         img = img.unsqueeze(0).to(self.device)  # добавляем batch dim и на устройство

#         pred = self.model(img, augment=False, visualize=False)

#         det = non_max_suppression(
#             pred,
#             conf_thres=self.conf,
#             iou_thres=0.45,
#             max_det=1000,
#             classes=None,
#             agnostic=False
#         )[0]

#         detections = []
#         if det is not None and len(det):
#             det = det.cpu()
#             for *xyxy, conf, cls in det:
#                 if int(cls) != 0:  # только люди (COCO class 0)
#                     continue
#                 x1, y1, x2, y2 = (int(x) for x in xyxy)
#                 detections.append([x1, y1, x2, y2, float(conf)])
#         return detections

# class NanoDetDetector(Detector):
#     def __init__(self, onnx_path, conf=0.5, input_size=416):
#         self.session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
#         self.input_name = self.session.get_inputs()[0].name
#         self.conf = conf
#         self.input_size = input_size

#     @staticmethod
#     def preprocess(frame, size):
#         img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         img_resized = cv2.resize(img, (size, size))
#         img_norm = img_resized.astype(np.float32) / 255.0
#         img_transposed = img_norm.transpose(2, 0, 1)[None, ...]
#         return img_transposed

#     @staticmethod
#     def postprocess(preds, original_shape, input_shape, conf_thresh=0.5):
#         boxes, scores, classes = preds
#         # boxes: [N, 4], scores: [N], classes: [N]
#         keep = scores >= conf_thresh
#         boxes = boxes[keep]
#         scores = scores[keep]
#         classes = classes[keep]

#         h, w = original_shape[:2]
#         ih, iw = input_shape

#         scale = min(iw / w, ih / h)
#         pad_w = (iw - w * scale) / 2
#         pad_h = (ih - h * scale) / 2

#         boxes[:, 0] = (boxes[:, 0] - pad_w) / scale
#         boxes[:, 2] = (boxes[:, 2] - pad_w) / scale
#         boxes[:, 1] = (boxes[:, 1] - pad_h) / scale
#         boxes[:, 3] = (boxes[:, 3] - pad_h) / scale

#         # clamp
#         boxes[:, 0] = np.clip(boxes[:, 0], 0, w)
#         boxes[:, 2] = np.clip(boxes[:, 2], 0, w)
#         boxes[:, 1] = np.clip(boxes[:, 1], 0, h)
#         boxes[:, 3] = np.clip(boxes[:, 3], 0, h)

#         detections = []
#         for (x1, y1, x2, y2), conf, cls in zip(boxes, scores, classes):
#             if int(cls) != 0:  # люди
#                 continue
#             detections.append([int(x1), int(y1), int(x2), int(y2), float(conf)])
#         return detections

#     def detect(self, frame):
#         orig_h, orig_w = frame.shape[:2]
#         inp = self.preprocess(frame, self.input_size)
#         outs = self.session.run(None, {self.input_name: inp})
#         # NanoDet ONNX обычно возвращает (boxes, scores, classes)
#         return self.postprocess(outs, (orig_h, orig_w), (self.input_size, self.input_size), self.conf)

# # class MMDetDetector(Detector):
# #     def __init__(self, config_path, checkpoint_path, conf=0.5, device=None):
# #         if device is None:
# #             device = 'cuda' if torch.cuda.is_available() else 'cpu'
# #         self.model = init_detector(config_path, checkpoint=checkpoint_path, device=device)
# #         self.conf = conf

# #     def detect(self, frame):
# #         result = inference_detector(self.model, frame)
# #         # result может быть tuple (boxes, labels) или просто boxes в зависимости от версии
# #         if isinstance(result, tuple):
# #             boxes, labels = result
# #         else:
# #             boxes = result

# #         detections = []
# #         # MMDet возвращает boxes [N, 5] (x1,y1,x2,y2,score) по классам
# #         # Для COCO class 0 — люди
# #         for i, bbox in enumerate(boxes[0]):  # boxes[class_id] -> [N,5]
# #             x1, y1, x2, y2, score = bbox
# #             if score < self.conf:
# #                 continue
# #             # class_id = i
# #             if i != 0:
# #                 continue  # только люди
# #             detections.append([int(x1), int(y1), int(x2), int(y2), float(score)])
# #         return detections


In [35]:
#  Фабрика детекторов

def create_detector(name, conf=0.5):
    if name == "yolov5n":
        return YOLOv5Detector("yolov5n.pt", conf=conf)
    elif name == "yolov5m":
        return YOLOv5Detector("yolov5m.pt", conf=conf)
    elif name == "nanodet_t":
        return NanoDetDetector("nanodet_tiny.onnx", conf=conf, input_size=416)
    # elif name == "faster_rcnn_r50":
    #     return MMDetDetector(
    #         "configs/faster_rcnn/faster-rcnn_r50_fpn_1x_coco.py",
    #         "faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth",
    #         conf=conf
    #     )
    else:
        raise ValueError(f"Unknown detector: {name}")

In [36]:
detector_name = "yolov5n"
detector = create_detector(detector_name)
print(
    "Loaded detector:",
    detector_name
)


# detector_name = "yolov8m"
# detector_name = "rtdetr"



PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Loaded detector: yolov5n


#3 — REID

In [37]:
import torch
import torchreid
import torchvision.transforms as T
from PIL import Image
import numpy as np

In [38]:
class ReIDModel:
    def get_embedding(self, image):
        raise NotImplementedError

In [39]:
class TorchReID(ReIDModel):

    def __init__(
        self,
        model_name,
        device=DEVICE
    ):
        self.device = device

        self.model = torchreid.models.build_model(
            name=model_name,
            num_classes=1000,
            pretrained=True
        )

        self.model.to(device)
        self.model.eval()

        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((256, 128)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    @torch.no_grad()
    def get_embedding(self, image):

        img = self.transform(image)

        img = img.unsqueeze(0).to(self.device)

        feature = self.model(img)

        feature = feature.cpu().numpy()[0]

        # L2 normalization
        feature /= np.linalg.norm(feature)

        return feature

In [40]:
def create_reid(name):

    if name == "osnet_x0_25":
        return TorchReID(
            "osnet_x0_25"
        )

    elif name == "osnet_x1_0":
        return TorchReID(
            "osnet_x1_0"
        )

    elif name == "resnet50":
        return TorchReID(
            "resnet50"
        )

    else:
        raise ValueError(
            f"Unknown REID model: {name}"
        )

In [41]:
reid_name = "osnet_x0_25"

reid = create_reid(reid_name)

print(
    "Loaded REID:",
    reid_name
)

Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
Loaded REID: osnet_x0_25


#4 — DeepSORT.

In [42]:
import sys
sys.path.append('./deep_sort')

from deep_sort import nn_matching
from deep_sort.detection import Detection
from deep_sort.tracker import Tracker

In [43]:
class DeepSORTWrapper:

    def __init__(
        self,
        max_cosine_distance=0.3,
        nn_budget=100
    ):

        self.metric = nn_matching.NearestNeighborDistanceMetric(
            "cosine",
            max_cosine_distance,
            nn_budget
        )

        self.tracker = Tracker(self.metric)
    def update(
        self,
        frame,
        detections,
        reid
    ):
        """
        detections:
        [x1, y1, x2, y2, confidence]
        """

        ds_detections = []


        for det in detections:

            x1, y1, x2, y2, conf = det


            # Вырезаем человека
            crop = frame[y1:y2, x1:x2]


            if crop.size == 0:
                continue


            # Получаем embedding
            feature = reid.get_embedding(crop)


            # DeepSORT использует tlwh
            bbox = [
                x1,
                y1,
                x2 - x1,
                y2 - y1
            ]


            ds_detections.append(
                Detection(
                    bbox,
                    conf,
                    feature
                )
            )
                    # Предсказание нового положения треков
        self.tracker.predict()


        # Сопоставление детекций и треков
        self.tracker.update(
            ds_detections
        )


        results = []


        for track in self.tracker.tracks:

            if not track.is_confirmed():
                continue


            if track.time_since_update > 1:
                continue


            bbox = track.to_tlbr()


            results.append({
                "id": track.track_id,
                "bbox": bbox
            })


        return results

In [44]:
tracker = DeepSORTWrapper(
    max_cosine_distance=0.3,
    nn_budget=100
)
detector = create_detector(
    "yolov5n"
)

reid = create_reid(
    "osnet_x0_25"
)

PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"


#5 — запуск трекинга и сохранение результатов

In [45]:
class MOTWriter:
    def __init__(self, filename):
        self.file = open(filename, "w")


    def write(self, frame_id, track_id, bbox):
        x1, y1, x2, y2 = bbox

        width = x2 - x1
        height = y2 - y1

        line = (
            f"{frame_id},{track_id},"
            f"{x1:.2f},{y1:.2f},"
            f"{width:.2f},{height:.2f},"
            "1,-1,-1,-1\n"
        )

        self.file.write(line)


    def close(self):
        self.file.close()

In [46]:
def draw_tracks(frame, tracks):

    for track in tracks:

        x1, y1, x2, y2 = map(int, track["bbox"])

        track_id = track["id"]

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"ID {track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

    return frame

In [47]:
import kagglehub

path = kagglehub.dataset_download(
    "takshmandar/mot16-dataset"
)

print(path)

Using Colab cache for faster access to the 'mot16-dataset' dataset.
/kaggle/input/mot16-dataset


In [48]:
for root, dirs, files in os.walk(path):
    print(root, dirs)
    print(files[:5])
    break

/kaggle/input/mot16-dataset ['test', 'train']
[]


In [49]:
mot09 = os.path.join(path, "train", "MOT16-09")

for item in os.listdir(mot09):
    print(item)

img1
gt
seqinfo.ini
det


In [50]:
img_dir = os.path.join(path, "train", "MOT16-09", "img1")

images = sorted(os.listdir(img_dir))
print("Количество кадров:", len(images))
print("Первые кадры:", images[:5])

Количество кадров: 525
Первые кадры: ['000001.jpg', '000002.jpg', '000003.jpg', '000004.jpg', '000005.jpg']


In [51]:
img_gt_dir = os.path.join(path, "train", "MOT16-09", "gt")
img_gt_dir

'/kaggle/input/mot16-dataset/train/MOT16-09/gt'

In [52]:
def run_sequence(
    img_dir,
    output_video,
    mot_path,
    detector,
    reid,
    tracker,
    fps=30
):

    frames = sorted(os.listdir(img_dir))

    first_frame = cv2.imread(os.path.join(img_dir, frames[0]))
    h, w = first_frame.shape[:2]

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    mot_writer = MOTWriter(mot_path)

    total_time = 0

    for i, frame_name in enumerate(frames, start=1):

        frame_path = os.path.join(img_dir, frame_name)
        frame = cv2.imread(frame_path)

        if frame is None:
            continue

        start = time.time()

        detections = detector.detect(frame)
        tracks = tracker.update(frame, detections, reid)

        elapsed = time.time() - start
        total_time += elapsed

        fps_now = 1 / elapsed if elapsed > 0 else 0

        cv2.putText(frame, f"FPS: {fps_now:.1f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

        frame = draw_tracks(frame, tracks)

        for t in tracks:
            mot_writer.write(i, t["id"], t["bbox"])

        writer.write(frame)

        if i % 50 == 0:
            print(f"Frame {i}/{len(frames)} FPS={fps_now:.2f}")

    writer.release()
    mot_writer.close()

    avg_fps = len(frames) / total_time if total_time > 0 else 0

    print("DONE")
    print("AVG FPS:", avg_fps)

    return avg_fps

In [53]:
avg_fps = run_sequence(
    img_dir=img_dir,
    output_video="results/MOT16-09.mp4",
    mot_path="results/MOT16-09.txt",
    detector=detector,
    reid=reid,
    tracker=tracker
)

Frame 50/525 FPS=3.52
Frame 100/525 FPS=2.89
Frame 150/525 FPS=3.36
Frame 200/525 FPS=2.54
Frame 250/525 FPS=2.39
Frame 300/525 FPS=2.96
Frame 350/525 FPS=3.46
Frame 400/525 FPS=2.76
Frame 450/525 FPS=2.08
Frame 500/525 FPS=2.74
DONE
AVG FPS: 2.6240934239157694


In [54]:
from google.colab import drive
drive.mount('/content/drive')

# Define a base directory in your Google Drive to save results
gdrive_output_dir = "/content/drive/MyDrive/deep_sort_results"
os.makedirs(gdrive_output_dir, exist_ok=True)
print(f"Google Drive output directory created: {gdrive_output_dir}")

Mounted at /content/drive
Google Drive output directory created: /content/drive/MyDrive/deep_sort_results


In [55]:
import csv
import pandas as pd

#6 — система перебора параметров

In [56]:
experiment_results = []

def run_experiment(
    video_path,
    detector_name,
    reid_name,
    conf,
    cosine_distance,
    nn_budget
):

    print("="*50)
    print(
        f"Detector: {detector_name}, "
        f"REID: {reid_name}"
    )

    print(
        f"conf={conf}, "
        f"cosine={cosine_distance}"
    )


    detector = create_detector(
        detector_name
    )

    detector.conf = conf


    reid = create_reid(
        reid_name
    )


    tracker = DeepSORTWrapper(
        max_cosine_distance=cosine_distance,
        nn_budget=nn_budget
    )


    output_video = (
        "results/temp.mp4"
    )

    output_mot = (
        "results/temp.txt"
    )


    fps = run_sequence(
        img_dir=video_path,       # <-- переименовал аргумент для ясности
        output_video=output_video,
        mot_path=output_mot,
        detector=detector,
        reid=reid,
        tracker=tracker
    )


    result = {
        "detector": detector_name,
        "reid": reid_name,
        "confidence": conf,
        "cosine_distance": cosine_distance,
        "nn_budget": nn_budget,
        "fps": fps
    }


    experiment_results.append(result)


    return result

# run_experiment(video_path=img_dir, detector_name="yolov8n", reid_name="osnet_x0_25", conf=0.5, cosine_distance=0.3, nn_budget=100)


In [57]:
detectors = [
    "yolov5n",
    "yolov5m",
    "nanodet_t",
    # "faster_rcnn_r50"
]


reids = [
    "osnet_x0_25",
    # "osnet_x1_0",
    # "resnet50"
]


confidences = [
    # 0.3,
    0.5,
    # 0.7
]


cosine_distances = [
#     0.2,
#     0.3,
    0.5
]


nn_budgets = [
    50,
    # 100
]

In [58]:
# for detector in detectors:

#     for reid in reids:

#         for conf in confidences:

#             for cosine in cosine_distances:

#                 for budget in nn_budgets:

#                     run_experiment(
#                         video_path=img_dir,
#                         detector_name=detector,
#                         reid_name=reid,
#                         conf=conf,
#                         cosine_distance=cosine,
#                         nn_budget=budget
#                     )

# df = pd.DataFrame(
#     experiment_results
# )

# df

In [59]:
# output_path = os.path.join(gdrive_output_dir, "metrics", "experiments.csv")
# os.makedirs(os.path.dirname(output_path), exist_ok=True)

# fieldnames = [
#     "detector", "reid", "conf", "cosine_distance", "nn_budget", 'fps', 'confidence'
#     # сюда добавь все ключи, которые возвращает metrics, например:
#     # "MOTA", "IDF1", "MT", "ML", "FP", "FN", "IDs" ==================================================

# ]

# file_exists = os.path.isfile(output_path)

# with open(output_path, mode="a", newline="", encoding="utf-8") as f:
#     writer = csv.DictWriter(f, fieldnames=fieldnames)
#     if not file_exists:
#         writer.writeheader()

#     for detector in detectors:
#         for reid in reids:
#             for conf in confidences:
#                 for cosine in cosine_distances:
#                     for budget in nn_budgets:
#                         try:
#                             metrics = run_experiment(
#                                 video_path=img_dir,
#                                 detector_name=detector,
#                                 reid_name=reid,
#                                 conf=conf,
#                                 cosine_distance=cosine,
#                                 nn_budget=budget
#                             )
#                             row = {
#                                 "detector": detector,
#                                 "reid": reid,
#                                 "conf": conf,
#                                 "cosine_distance": cosine,
#                                 "nn_budget": budget,
#                                 **metrics
#                             }
#                             writer.writerow(row)
#                             f.flush()          # принудительно сбрасываем на диск
#                             os.fsync(f.fileno())  # гарантия записи на диск (для Linux/Colab)
#                             print(f"Appended: detector={detector}, reid={reid}, conf={conf}")

#                         except Exception as e:
#                             print(f"Failed: detector={detector}, reid={reid}, conf={conf}, cosine={cosine}, budget={budget}")
#                             print("Error:", e)

Detector: yolov5n, REID: osnet_x0_25
conf=0.5, cosine=0.5
PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"


KeyboardInterrupt: 

In [ ]:
# df.to_csv(
#     "metrics/experiments.csv",
#     index=False
# )


# os.makedirs(os.path.join(gdrive_output_dir, "metrics"), exist_ok=True)
# df.to_csv(
#     os.path.join(gdrive_output_dir, "metrics", "experiments.csv"),
#     index=False
# )

7 — оценку качества трекинга

In [ ]:
!git clone https://github.com/JonathonLuiten/TrackEval.git
# %cd TrackEval
# !pip install -r requirements.txt
# %cd ..
!pip install "numpy<2.0"
!pip install pandas scipy matplotlib seaborn tqdm

In [ ]:
import motmetrics as mm
import pandas as pd

def evaluate_mot(gt_file, pred_file):
    gt = mm.io.loadtxt(gt_file, fmt="mot15-2D")
    pred = mm.io.loadtxt(pred_file, fmt="mot15-2D")

    acc = mm.utils.compare_to_groundtruth(gt, pred)

    metrics = [
        "mota",
        "idf1",
        "precision",
        "recall",
        "num_switches"
    ]

    mh = mm.metrics.create()
    summary = mh.compute(
        acc,
        metrics=metrics,
        name="tracking"
    )

    return summary


In [ ]:
!python TrackEval/scripts/run_mot_challenge.py \
    --BENCHMARK MOT16 \
    --TRACKERS_TO_EVAL our_tracker \
    --METRICS HOTA

# 8 -

In [ ]:
MOT_SEQUENCES = [
    "TUD-Campus",
    "TUD-Stadtmitte",
    "KITTI-17",
    "PETS09-S2L1",
    "MOT16-09",
    "MOT16-11"
]